# 01 — Data Ingestion and Validation

## Purpose

This notebook is the first executable step of the E-Commerce Intelligence Platform.

It will:

1. Validate that the original Olist CSV files exist.
2. Load the raw CSV files without modifying them.
3. Inspect schema, row counts, missing values, and duplicates.
4. Validate important relational keys.
5. Connect to Supabase PostgreSQL.
6. Create the raw database tables from `sql/schema.sql` if they do not exist.
7. Load the validated raw tables into Supabase.
8. Verify database row counts.

> **Important:** This notebook does not perform feature engineering, analytical merging, or machine learning. The raw CSV files remain the source data.

## Expected project structure

```text
ecommerce-intelligence/
├── data/
│   └── raw/
│       ├── olist_customers_dataset.csv
│       ├── olist_geolocation_dataset.csv
│       ├── olist_order_items_dataset.csv
│       ├── olist_order_payments_dataset.csv
│       ├── olist_order_reviews_dataset.csv
│       ├── olist_orders_dataset.csv
│       ├── olist_products_dataset.csv
│       ├── olist_sellers_dataset.csv
│       └── product_category_name_translation.csv
│
├── sql/
│   └── schema.sql
│
├── src/
│   └── database.py
│
└── .env
```

Run this notebook from the project root.

In [5]:
from pathlib import Path
import sys
import logging

import pandas as pd
from sqlalchemy import text


# ============================================================
# PROJECT PATH
# ============================================================

# Current working directory used by VS Code/Jupyter
CURRENT_DIR = Path.cwd()

# If notebook is running from notebooks/, project root is its parent.
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Make project root importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# IMPORT PROJECT MODULES
# ============================================================

from src.database import get_engine, get_connection


# ============================================================
# CONFIGURATION
# ============================================================

logging.basicConfig(level=logging.INFO)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SCHEMA_FILE = PROJECT_ROOT / "sql" / "schema.sql"


# ============================================================
# VERIFY PATHS
# ============================================================

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Schema file: {SCHEMA_FILE}")

print("\nProject root exists:", PROJECT_ROOT.exists())
print("src directory exists:", (PROJECT_ROOT / "src").exists())
print("database.py exists:", (PROJECT_ROOT / "src" / "database.py").exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Schema file exists:", SCHEMA_FILE.exists())

Project root: d:\ecommerce-intelligence
Raw data directory: d:\ecommerce-intelligence\data\raw
Schema file: d:\ecommerce-intelligence\sql\schema.sql

Project root exists: True
src directory exists: True
database.py exists: True
Raw data directory exists: True
Schema file exists: True


## 1. Validate required files

In [6]:
EXPECTED_FILES = {
    "olist_customers": "olist_customers_dataset.csv",
    "olist_geolocation": "olist_geolocation_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
    "olist_orders": "olist_orders_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv",
}

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Raw data directory does not exist: {RAW_DIR}"
    )

missing_files = [
    filename for filename in EXPECTED_FILES.values()
    if not (RAW_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following required Olist files are missing:\n"
        + "\n".join(f"- {name}" for name in missing_files)
    )

if not SCHEMA_FILE.is_file():
    raise FileNotFoundError(f"Database schema not found: {SCHEMA_FILE}")

print(f"All {len(EXPECTED_FILES)} required Olist files are present.")
print("Database schema file is present.")

All 9 required Olist files are present.
Database schema file is present.


## 2. Load the original CSV files

In [7]:
dataframes = {}

for table_name, filename in EXPECTED_FILES.items():
    path = RAW_DIR / filename
    df = pd.read_csv(path, low_memory=False)
    dataframes[table_name] = df
    print(f"{table_name:40s} {df.shape}")

olist_customers                          (99441, 5)
olist_geolocation                        (1000163, 5)
olist_order_items                        (112650, 7)
olist_order_payments                     (103886, 5)
olist_order_reviews                      (99224, 7)
olist_orders                             (99441, 8)
olist_products                           (32951, 9)
olist_sellers                            (3095, 4)
product_category_name_translation        (71, 2)


## 3. Inspect columns and data types

In [8]:
for table_name, df in dataframes.items():
    print("\n" + "=" * 80)
    print(table_name)
    print("=" * 80)
    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values,
            "missing": df.isna().sum().values,
            "missing_pct": (df.isna().mean() * 100).round(2).values,
        })
    )


olist_customers


,column,dtype,missing,missing_pct
0,customer_id,str,0,0.0
1,customer_unique_id,str,0,0.0
2,customer_zip_code_prefix,int64,0,0.0
3,customer_city,str,0,0.0
4,customer_state,str,0,0.0



olist_geolocation


,column,dtype,missing,missing_pct
0,geolocation_zip_code_prefix,int64,0,0.0
1,geolocation_lat,float64,0,0.0
2,geolocation_lng,float64,0,0.0
3,geolocation_city,str,0,0.0
4,geolocation_state,str,0,0.0



olist_order_items


...

,column,dtype,missing,missing_pct
0,product_id,str,0,0.00
1,product_category_name,str,610,1.85
2,product_name_lenght,float64,610,1.85
3,product_description_lenght,float64,610,1.85
4,product_photos_qty,float64,610,1.85
5,product_weight_g,float64,2,0.01
6,product_length_cm,float64,2,0.01
7,product_height_cm,float64,2,0.01
8,product_width_cm,float64,2,0.01



olist_sellers


,column,dtype,missing,missing_pct
0,seller_id,str,0,0.0
1,seller_zip_code_prefix,int64,0,0.0
2,seller_city,str,0,0.0
3,seller_state,str,0,0.0



product_category_name_translation


,column,dtype,missing,missing_pct
0,product_category_name,str,0,0.0
1,product_category_name_english,str,0,0.0


## 4. Check duplicate records

In [9]:
duplicate_summary = []

for table_name, df in dataframes.items():
    duplicate_summary.append({
        "table": table_name,
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
    })

duplicate_summary = pd.DataFrame(duplicate_summary)
display(duplicate_summary)

print("Note: duplicate rows are reported, not automatically removed.")
print("The raw source files remain immutable.")

,table,rows,duplicate_rows
0,olist_customers,99441,0
1,olist_geolocation,1000163,261831
2,olist_order_items,112650,0
3,olist_order_payments,103886,0
4,olist_order_reviews,99224,0
5,olist_orders,99441,0
6,olist_products,32951,0
7,olist_sellers,3095,0
8,product_category_name_translation,71,0


Note: duplicate rows are reported, not automatically removed.
The raw source files remain immutable.


## 5. Validate important key columns

The checks below focus on the identifiers that define the Olist relational structure.

We intentionally do not require every identifier to be globally unique:

- `customer_id` is unique in the customer table.
- `order_id` is unique in the orders table.
- `product_id` is unique in the products table.
- `seller_id` is unique in the sellers table.
- `order_id + order_item_id` identifies an order-item row.
- `order_id + payment_sequential` identifies a payment row.
- The geolocation table is **not** expected to have a unique ZIP prefix.

In [10]:
def uniqueness_check(df, columns):
    return {
        "columns": ", ".join(columns),
        "rows": len(df),
        "unique_keys": int(df[list(columns)].drop_duplicates().shape[0]),
        "duplicate_keys": int(df.duplicated(subset=list(columns)).sum()),
    }

key_checks = [
    ("olist_customers", ["customer_id"]),
    ("olist_orders", ["order_id"]),
    ("olist_products", ["product_id"]),
    ("olist_sellers", ["seller_id"]),
    ("olist_order_items", ["order_id", "order_item_id"]),
    ("olist_order_payments", ["order_id", "payment_sequential"]),
    ("olist_order_reviews", ["review_id", "order_id"]),
]

key_results = []

for table_name, columns in key_checks:
    result = uniqueness_check(dataframes[table_name], columns)
    result["table"] = table_name
    key_results.append(result)

key_results = pd.DataFrame(key_results)
display(key_results)

,columns,rows,unique_keys,duplicate_keys,table
0,customer_id,99441,99441,0,olist_customers
1,order_id,99441,99441,0,olist_orders
2,product_id,32951,32951,0,olist_products
3,seller_id,3095,3095,0,olist_sellers
4,"order_id, order_item_id",112650,112650,0,olist_order_items
5,"order_id, payment_sequential",103886,103886,0,olist_order_payments
6,"review_id, order_id",99224,99224,0,olist_order_reviews


## 6. Validate foreign-key-style relationships in the raw data

In [11]:
def missing_references(child_df, child_key, parent_df, parent_key):
    child_values = child_df[child_key].dropna().unique()
    parent_values = set(parent_df[parent_key].dropna().unique())
    return sum(value not in parent_values for value in child_values)

relationship_checks = [
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_orders"], "customer_id",
            dataframes["olist_customers"], "customer_id"
        ),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_order_items"], "order_id",
            dataframes["olist_orders"], "order_id"
        ),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_order_items"], "product_id",
            dataframes["olist_products"], "product_id"
        ),
    },
    {
        "relationship": "order_items.seller_id -> sellers.seller_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_order_items"], "seller_id",
            dataframes["olist_sellers"], "seller_id"
        ),
    },
    {
        "relationship": "order_payments.order_id -> orders.order_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_order_payments"], "order_id",
            dataframes["olist_orders"], "order_id"
        ),
    },
    {
        "relationship": "order_reviews.order_id -> orders.order_id",
        "missing_parent_keys": missing_references(
            dataframes["olist_order_reviews"], "order_id",
            dataframes["olist_orders"], "order_id"
        ),
    },
]

relationship_results = pd.DataFrame(relationship_checks)
display(relationship_results)

,relationship,missing_parent_keys
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0
3,order_items.seller_id -> sellers.seller_id,0
4,order_payments.order_id -> orders.order_id,0
5,order_reviews.order_id -> orders.order_id,0


## 7. Inspect timestamp columns

Timestamp parsing is performed on copies for validation only.

The original CSV files are not modified.

In [12]:
timestamp_columns = {
    "olist_orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "olist_order_items": ["shipping_limit_date"],
    "olist_order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

timestamp_summary = []

for table_name, columns in timestamp_columns.items():
    df = dataframes[table_name]
    for column in columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        timestamp_summary.append({
            "table": table_name,
            "column": column,
            "non_null": int(parsed.notna().sum()),
            "invalid_or_unparseable": int(
                df[column].notna().sum() - parsed.notna().sum()
            ),
            "min": parsed.min(),
            "max": parsed.max(),
        })

timestamp_summary = pd.DataFrame(timestamp_summary)
display(timestamp_summary)

,table,column,non_null,invalid_or_unparseable,min,max
0,olist_orders,order_purchase_timestamp,99441,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,olist_orders,order_approved_at,99281,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,olist_orders,order_delivered_carrier_date,97658,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,olist_orders,order_delivered_customer_date,96476,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,olist_orders,order_estimated_delivery_date,99441,0,2016-09-30 00:00:00,2018-11-12 00:00:00
5,olist_order_items,shipping_limit_date,112650,0,2016-09-19 00:15:34,2020-04-09 22:35:08
6,olist_order_reviews,review_creation_date,99224,0,2016-10-02 00:00:00,2018-08-31 00:00:00
7,olist_order_reviews,review_answer_timestamp,99224,0,2016-10-07 18:32:28,2018-10-29 12:27:35


## 8. Inspect categorical domains

This provides a quick source-data sanity check before database ingestion.

In [13]:
categorical_columns = {
    "olist_orders": ["order_status"],
    "olist_order_payments": ["payment_type"],
    "olist_order_reviews": ["review_score"],
    "olist_customers": ["customer_state"],
    "olist_sellers": ["seller_state"],
}

for table_name, columns in categorical_columns.items():
    print("\n" + table_name)
    for column in columns:
        print(f"{column}:")
        print(dataframes[table_name][column].value_counts(dropna=False).head(20))


olist_orders
order_status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

olist_order_payments
payment_type:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

olist_order_reviews
review_score:
review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

olist_customers
customer_state:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
Name: count, dtype: int64

olist_sellers
seller_state:
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
B

## 9. Connect to Supabase PostgreSQL

Before executing this section:

- Create the Supabase project.
- Add the real credentials to `.env`.
- Make sure the database is reachable.

The connection uses `src/database.py` so credentials remain outside this notebook.

In [14]:
engine = get_engine()

with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database(), current_user"))
    database_name, database_user = result.fetchone()

print(f"Connected to database: {database_name}")
print(f"Connected as: {database_user}")

Connected to database: postgres
Connected as: postgres


## 10. Create the raw database schema

This cell executes `sql/schema.sql`.

It is safe to rerun because the schema uses `CREATE TABLE IF NOT EXISTS` and `CREATE INDEX IF NOT EXISTS`.

No data is loaded by this cell.

In [ ]:
schema_sql = SCHEMA_FILE.read_text(encoding="utf-8")

# Execute individual SQL statements while ignoring comments and blank lines.
# The schema file contains straightforward CREATE TABLE / CREATE INDEX
# statements without procedural PostgreSQL blocks.
statements = [
    statement.strip()
    for statement in schema_sql.split(";")
    if statement.strip()
]

with engine.begin() as connection:
    for statement in statements:
        if statement and not statement.startswith("--"):
            connection.execute(text(statement))

print(f"Executed {len(statements)} schema statements.")

ProgrammingError: (psycopg2.ProgrammingError) can't execute an empty query
[SQL: -- ============================================================
-- End of schema
-- ============================================================]
(Background on this error at: https://sqlalche.me/e/20/f405)

## 11. Load validated raw data into Supabase

In [ ]:
# The database column names match the original CSV column names.
# SQLAlchemy/Pandas performs the insertion without changing the raw
# dataframe structure.

for table_name, df in dataframes.items():
    print(f"Loading {table_name}: {len(df):,} rows")

    df.to_sql(
        table_name,
        con=engine,
        schema="public",
        if_exists="append",
        index=False,
        chunksize=5000,
        method="multi",
    )

    print(f"Finished {table_name}")

### Important ingestion note

This notebook uses `append` because the source tables are expected to be empty after the initial schema creation.

If you rerun the complete ingestion cell after data has already been inserted, primary-key conflicts or duplicate rows can occur.

For a clean re-ingestion, truncate the raw tables first or recreate the database schema. Do **not** delete or modify the original CSV files.

## 12. Verify database row counts

In [ ]:
database_counts = []

with engine.connect() as connection:
    for table_name in EXPECTED_FILES:
        count = connection.execute(
            text(f'SELECT COUNT(*) FROM public."{table_name}"')
        ).scalar_one()

        source_count = len(dataframes[table_name])

        database_counts.append({
            "table": table_name,
            "source_rows": source_count,
            "database_rows": int(count),
            "match": source_count == int(count),
        })

database_counts = pd.DataFrame(database_counts)
display(database_counts)

if not database_counts["match"].all():
    raise AssertionError("At least one database row count does not match the source CSV.")

print("All source and database row counts match.")

## 13. Final ingestion validation

In [ ]:
# Re-run key relationship checks against the database.

validation_queries = {
    "customers": 'SELECT COUNT(*) FROM public."olist_customers"',
    "orders": 'SELECT COUNT(*) FROM public."olist_orders"',
    "order_items": 'SELECT COUNT(*) FROM public."olist_order_items"',
    "products": 'SELECT COUNT(*) FROM public."olist_products"',
    "sellers": 'SELECT COUNT(*) FROM public."olist_sellers"',
    "payments": 'SELECT COUNT(*) FROM public."olist_order_payments"',
    "reviews": 'SELECT COUNT(*) FROM public."olist_order_reviews"',
    "geolocation": 'SELECT COUNT(*) FROM public."olist_geolocation"',
    "category_translation": 'SELECT COUNT(*) FROM public."product_category_name_translation"',
}

with engine.connect() as connection:
    final_counts = {
        name: connection.execute(text(query)).scalar_one()
        for name, query in validation_queries.items()
    }

display(pd.DataFrame(
    [{"table": name, "rows": count} for name, count in final_counts.items()]
))

print("Database ingestion and validation completed successfully.")

# Conclusion

The raw Olist relational dataset is now validated and, once the Supabase credentials are configured, can be loaded into PostgreSQL without pre-merging the source tables.

The next notebook will use:

- Supabase SQL
- Python
- EDA
- Statistics
- Plotly/Matplotlib

to investigate customers, orders, products, categories, sellers, payments, reviews, delivery, and geography.

**No machine-learning features are created in this notebook.**